# HARP-PCB v0.4 — Functional roles + optional RF keepouts

Upload an **unrouted** `.kicad_pcb` with overlapping or pre-placed parts. The placer rebuilds a rectangular board, recommends I/O near edges and a recognized processor near the center, and saves intermediate placement snapshots.

**Antenna placement:** a recognized antenna gets a soft edge preference and a soft separation from detectable switching-noise sources and processors. If your antenna datasheet specifies a minimum clearance, enter that as a HARD `antenna_keepouts` rule below. Do not assume any default spacing is RF-safe. Copper, ground-plane voids and matching-network geometry are not generated or validated.

The optional IntelCAD LLM diagnoses placeholder footprints but does not directly move parts or invent an RF keepout. This is **geometry-only output**, not a DRC-passed routed board.


## 1 · Settings and optional IntelCAD API key

- The API key is entered through a masked prompt. Leave empty to run **without AI**, using the local safe fallback. If enabled, only flagged **reference, footprint identifier, value and pad count** are transmitted to IntelCAD; **not raw PCB, netlist, pad coordinates, or output file**.
- `STRICT_PREFLIGHT` restores the previous refusal behavior for unverified footprints/ambiguous orientations. Default `False` keeps placing while reporting warnings.
- `SHOW_STAGES` saves snapshots and allows choosing an intermediate step.


In [ ]:
# Colab configuration
SHOW_STAGES = True
SNAPSHOT_EVERY = 2000
ITERATIONS = 4000
SEED = 42
CLEARANCE_MM = 0.65
EDGE_MARGIN_MM = 2.0
STRICT_PREFLIGHT = False
ALLOW_REPLACE_NONRECTANGULAR_OUTLINE = False

# LLM is advisory ONLY: it does not modify footprints, pads, or board geometry.
USE_LLM_PREFLIGHT = True
USE_LLM_REPORT = False
MODEL = "cc/claude-haiku-5-5"  # optional: "cc/claude-opus-5-5"
BASE_URL = "https://ai-router.intelcad.ai/v1"
from getpass import getpass
AI_API_KEY = getpass("IntelCAD API key (optional, Enter to skip): ").strip() if USE_LLM_PREFLIGHT or USE_LLM_REPORT else ""
print("Preflight AI:", "enabled" if USE_LLM_PREFLIGHT and AI_API_KEY else "local fallback without API")


## 2 · Clone the public repository and install dependencies

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = Path("/content/PCB_placement")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/amirhossein-alishahi/PCB_placement.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[preview]"], check=True)
subprocess.run([sys.executable, "-m", "harp_pcb", "--help"], check=True, stdout=subprocess.DEVNULL)
print("Installed from:", REPO)

## 3 · Upload `.kicad_pcb` for preflight inspection

The initial report lists synthetic/placeholder footprints and any unresolved USB/SD/switch orientation. These are **warnings in best-effort mode**, not a reason to halt geometric placement. Mechanical opening directions still require a verified marker or explicit rule.


In [ ]:
from google.colab import files
import json, shutil
from pathlib import Path
uploaded = files.upload()
boards = [name for name in uploaded if name.lower().endswith('.kicad_pcb')]
if len(boards) != 1:
    raise ValueError("Upload exactly one .kicad_pcb file")
WORK = Path('/content/harp_job')
WORK.mkdir(parents=True, exist_ok=True)
SOURCE = WORK/'input.kicad_pcb'
shutil.copyfile(Path(boards[0]),SOURCE)
print('Selected:', boards[0], '| bytes:', SOURCE.stat().st_size)

## 4 · Configure opening directions and functional/RF rules

- `EDGE_RULES`: for **mechanically access-facing** connectors, specify footprint-local opening direction and desired edge; a soft I/O role alone cannot guarantee access.
- `ROLE_OVERRIDES`: optional manual assignments: `io`, `processor`, `core`, `antenna`, `noisy`, `general`. Otherwise the algorithm uses cautious name/value heuristics.
- `ANTENNA_KEEPOUTS`: **hard minimum mm between footprint envelopes**, set only after consulting your specific antenna/module manufacturer. `against='noisy'` includes processor/switching components and named references, `all` means all parts, and `listed` only the explicit `references`.

If you have an antenna but no verified clearance measurement, leave `ANTENNA_KEEPOUTS` empty. The run continues using soft preferences and records an RF review warning; it does not claim RF safety.


In [ ]:
EDGE_RULES = {
    # "J1": {"side": "right", "outward_local": "right", "edge_offset_mm": 0.5},
    # "J2": {"side": "bottom", "outward_local": "right", "edge_offset_mm": 0.5},
    # "SW1": {"side": "any", "outward_local": "top", "edge_offset_mm": 0.5},
}
# Optional soft functional classifications; use references that exist on your board.
ROLE_OVERRIDES = {
    # "J1": {"role": "io"},
    # "U1": {"role": "processor"},
    # "ANT1": {"role": "antenna"},
    # "U2": {"role": "noisy"},
}
# Only specify antenna spacing after checking the antenna/module datasheet.
# The number below is an EXAMPLE, not an antenna design recommendation.
ANTENNA_KEEPOUTS = {
    # "ANT1": {"distance_mm": 8.0, "against": "noisy", "references": ["U2"]},
}
PLACEMENT_RULE_FILE = WORK/'placement_rules.json'
PLACEMENT_RULE_FILE.write_text(json.dumps({
    'components':ROLE_OVERRIDES, 'antenna_keepouts':ANTENNA_KEEPOUTS
}, indent=2),encoding='utf-8')
RULE_FILE = WORK/'edge_rules.json'
RULE_FILE.write_text(json.dumps({"components": EDGE_RULES}, indent=2), encoding='utf-8')
PREFLIGHT_FILE = WORK/'preflight.json'
pre = subprocess.run([sys.executable,'-m','harp_pcb',str(SOURCE),'--direction-rules',str(RULE_FILE),
                      '--placement-rules',str(PLACEMENT_RULE_FILE),
                      '--preflight-only','--report',str(PREFLIGHT_FILE)],
                     capture_output=True, text=True)
print(pre.stdout if pre.returncode == 0 else pre.stderr)
if pre.returncode:
    raise RuntimeError('Preflight failed — inspect input board and edge rules')
PREFLIGHT = json.loads(PREFLIGHT_FILE.read_text())
print('Synthetic:',len(PREFLIGHT['synthetic_footprints']),
      '| Unresolved access-direction:',PREFLIGHT['unresolved_directions'])
print("Inferred roles:", {ref:x["role"] for ref,x in PREFLIGHT["functional_placement"]["roles"].items() if x["role"]!="general"})
print("RF review needed:", PREFLIGHT["functional_placement"]["rf_review_needed"])


## 5 · Best-effort placement, optional AI footprint review

Placement runs even if synthetic footprints or unknown connector directions remain. If the AI call fails, the CLI uses local fallback automatically. **The generated PCB is a geometry-only draft**; unverified parts are still unverified and all safety warnings remain in the JSON report. The run can still stop on potentially destructive inputs (existing traces/zones, complex outlines, invalid geometry).


In [ ]:
OUTPUT = WORK/'placed.kicad_pcb'
REPORT = WORK/'placement_report.json'
PREVIEW = WORK/'placed_preview.png'
STAGES = WORK/'stages'
cmd = [sys.executable,'-m','harp_pcb',str(SOURCE),'-o',str(OUTPUT),
       '--iterations',str(ITERATIONS),'--seed',str(SEED),
       '--clearance',str(CLEARANCE_MM),'--edge-margin',str(EDGE_MARGIN_MM),
       '--direction-rules',str(RULE_FILE),
       '--placement-rules',str(PLACEMENT_RULE_FILE),
       '--report',str(REPORT),'--preview',str(PREVIEW)]
if SHOW_STAGES:
    cmd += ['--stages-dir',str(STAGES),'--snapshot-every',str(SNAPSHOT_EVERY)]
if STRICT_PREFLIGHT:
    cmd.append('--strict-preflight')
if ALLOW_REPLACE_NONRECTANGULAR_OUTLINE:
    cmd.append('--replace-nonrectangular-outline')
if USE_LLM_PREFLIGHT:
    cmd += ['--ai-preflight','--ai-model',MODEL,'--ai-base-url',BASE_URL]
# Key is transferred via this process environment, NEVER by CLI arguments or files.
env = os.environ.copy()
if AI_API_KEY:
    env['INTELCAD_API_KEY'] = AI_API_KEY
run = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(run.stdout[-6000:] if run.returncode == 0 else run.stderr[-4000:])
if run.returncode:
    raise RuntimeError('The PCB has an unsafe/unsupported structural or geometrical condition. See error above.')
RESULT = json.loads(REPORT.read_text())
REVIEW = RESULT['footprint_audit']['ai_review']
print('Output:', OUTPUT, 'size:', OUTPUT.stat().st_size, 'bytes')
print('AI status:', REVIEW['status'], '| advisory recommendations:', len(REVIEW['recommendations']))
print('Manual review required:', RESULT['review_required_components'])
for warning in RESULT['footprint_audit']['nonblocking_warnings']:
    print('WARNING:', warning)
for rec in REVIEW['recommendations']:
    print('ADVISORY:',rec['ref'],rec['diagnosis'], '| Candidate:',rec['suggested_library_footprint'],
          '| Verify:',rec['verification_steps'])

print("Checked hard antenna keepouts:", RESULT.get("validated_rf_keepouts", []))


## 6 · View final and intermediate stages

In [ ]:
from IPython.display import display, Image, Markdown
if PREVIEW.exists():
    display(Image(filename=str(PREVIEW)))
if SHOW_STAGES and (STAGES/'manifest.json').exists():
    manifest = json.loads((STAGES/'manifest.json').read_text())
    print('Saved stages:', [m['name'] for m in manifest])
    import ipywidgets as widgets
    stage_choice = widgets.Dropdown(options=[(f"{i+1}: {m['name']}", i) for i,m in enumerate(manifest)],
                                    description='Stage:')
    out = widgets.Output()
    def change_stage(change=None):
        with out:
            out.clear_output(wait=True)
            chosen=manifest[stage_choice.value]
            display(Markdown(f"**{chosen['name']}** · `{chosen['pcb']}`"))
            display(Image(filename=str(STAGES/chosen['preview'])))
    stage_choice.observe(change_stage, names='value')
    display(stage_choice,out)
    change_stage()

## 7 · Optional Claude explanation of numerical metrics

This second, optional request only transmits aggregate statistics. The **preflight** AI advice above is already included in `placement_report.json`. Neither AI call modifies geometry.


In [ ]:
if USE_LLM_REPORT and AI_API_KEY:
    import urllib.request, urllib.error
    aggregate = {k: RESULT.get(k) for k in (
        'footprints','pads','nets','board_width_mm','board_height_mm','board_area_mm2',
        'min_bounding_box_gap_mm','runtime_seconds','accepted_moves')}
    aggregate['synthetic_count'] = len(RESULT['footprint_audit']['synthetic_footprints'])
    aggregate['unresolved_orientation_count'] = len(RESULT['footprint_audit']['unresolved_directions'])
    messages = [{'role':'system','content': 'Explain a PCB geometry-only placement report. Do not claim manufacturability, KiCad DRC, correct orientation, or routability. Reply in Persian.'},
                {'role':'user','content': json.dumps(aggregate)}]
    body=json.dumps({'model':MODEL,'messages':messages,'temperature':0.1,'max_tokens':700}).encode()
    req=urllib.request.Request(BASE_URL.rstrip('/')+'/chat/completions',data=body,method='POST',
              headers={'Authorization':'Bearer '+AI_API_KEY,'Content-Type':'application/json'})
    try:
        with urllib.request.urlopen(req,timeout=75) as res:
            answer=json.load(res)
        print(answer['choices'][0]['message']['content'])
    except (urllib.error.HTTPError,urllib.error.URLError,KeyError,ValueError) as exc:
        print('AI report unavailable. Placement is unaffected:',type(exc).__name__)
else:
    print('Skipped optional metrics-only LLM report (disabled or no key).')

## 8 · Download results and safety warnings

Download the geometry-only `.kicad_pcb`, stages, preflight data and AI advisories. A synthetic footprint stays synthetic even if the model suggests a replacement name. Confirm pad geometry and manufacturer datasheets, check the opening direction, run KiCad DRC and detailed routing before manufacture.


In [ ]:
from google.colab import files
import zipfile
ARCHIVE = WORK/'harp_results.zip'
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in [OUTPUT,REPORT,PREVIEW,PREFLIGHT_FILE,RULE_FILE,PLACEMENT_RULE_FILE]:
        if file.exists():
            z.write(file,arcname=file.name)
    if SHOW_STAGES and STAGES.exists():
        for file in STAGES.rglob('*'):
            if file.is_file():
                z.write(file,arcname='stages/'+str(file.relative_to(STAGES)))
print('Archive:', ARCHIVE,'size:',ARCHIVE.stat().st_size,'bytes')
files.download(str(ARCHIVE))